# (If using Colab)Access our github repo

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!git clone https://github.com/rohankumar-1/multimodal-seizure-detection.git

Cloning into 'multimodal-seizure-detection'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 58 (delta 27), reused 41 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 21.79 KiB | 3.11 MiB/s, done.
Resolving deltas: 100% (27/27), done.


In [3]:
import os

os.chdir("./multimodal-seizure-detection")

In [4]:
# try to update to make sure we are up to date
!git fetch

# Load in data, and apply preprocessing as necessary

In [1]:
import numpy as np

# train_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/train_data.npz", allow_pickle=True)
# val_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/val_data.npz", allow_pickle=True)
# test_data = np.load("/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/test_data.npz", allow_pickle=True)

train_data = np.load("data/train_data.npz", allow_pickle=True)
val_data = np.load("data/val_data.npz", allow_pickle=True)
test_data = np.load("data/test_data.npz", allow_pickle=True)


In [2]:
train_eeg, train_ecg = train_data['eeg'], train_data['ecg']
val_eeg, val_ecg = val_data['eeg'], val_data['ecg']
test_eeg, test_ecg = test_data['eeg'], test_data['ecg']

In [3]:
# this function returns a preprocessed tensor for the given modality, with filtering + normalization
# Parameters:
#   desired_fs: frequency of returned signal
#   lowcut: lower cut of the bandwith filter
#   highcut: higher cut of the bandwith filter
# output is tuple of tensors
from src import preprocess_signal_nn

train_eeg, val_eeg, test_eeg = preprocess_signal_nn(train_eeg, val_eeg, test_eeg, desired_fs=128, lowcut=0.5, highcut=60.0, notch_freq=50.0)
train_ecg, val_ecg, test_ecg = preprocess_signal_nn(train_ecg, val_ecg, test_ecg, desired_fs=128, lowcut=0.5, highcut=60.0, notch_freq=50.0)

In [4]:
import torch
from src import SupervisedMultimodalDataset

trainset = SupervisedMultimodalDataset({'eeg': train_eeg, 'ecg':train_ecg}, torch.tensor(train_data['binary_label']))
valset = SupervisedMultimodalDataset({'eeg': val_eeg, 'ecg': val_ecg}, torch.tensor(val_data['binary_label']))
testset = SupervisedMultimodalDataset({'eeg': test_eeg, 'ecg': test_ecg}, torch.tensor(test_data['binary_label']))

In [5]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=8, shuffle=True)
valloader = DataLoader(valset, batch_size=8, shuffle=False)
testloader = DataLoader(testset, batch_size=8, shuffle=False)

# SVM

In [32]:
# SVM Model
from src.models import SVMModel


train_eeg_flat = train_eeg.view(train_eeg.size(0), -1)
test_eeg_flat = test_eeg.view(test_eeg.size(0), -1)


m = SVMModel(kernel='linear')

In [33]:
import torch

N_samples = 1000
indices = torch.randperm(train_eeg_flat.size(0))[:N_samples]

train_eeg_flat_subsample = train_eeg_flat[indices]

In [34]:
m.fit(train_eeg_flat_subsample, train_data['binary_label'][indices])

In [35]:
m.evaluate(test_eeg_flat, test_data['binary_label'])

AUC: 0.5052
Accuracy: 0.7638
F1 Score: 0.7440
Precision: 0.7318
Recall: 0.7638


{'auc': 0.5052229917799048,
 'accuracy': 0.7638204225352113,
 'f1': 0.743984435122395,
 'precision': 0.7317662702979159,
 'recall': 0.7638204225352113}

# Basic Fusion

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TwoModalityMLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        # EEG: (B, 2, 2560) → flatten → (B, 5120)
        self.eeg_lin = nn.Linear(2 * 1280, hidden)

        # ECG: (B, 1, 2560)
        self.ecg_conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)   # → (B, 32, 1)
        )
        self.ecg_lin = nn.Linear(32, hidden)

        # Fusion MLP
        self.fuse = nn.Sequential(
            nn.ReLU(),
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, **kwargs):
        eeg = kwargs["eeg"]   # (B, 2, 2560)
        ecg = kwargs["ecg"]   # (B, 1, 2560)

        # EEG flatten
        eeg_flat = eeg.reshape(eeg.size(0), -1)  # (B, 5120)
        h1 = F.relu(self.eeg_lin(eeg_flat))

        # ECG conv
        ecg_feat = self.ecg_conv(ecg).squeeze(-1)  # (B, 32)
        h2 = F.relu(self.ecg_lin(ecg_feat))

        # Fusion
        h = torch.cat([h1, h2], dim=1)
        return self.fuse(h)

In [16]:
from src import train_supervised_nn

model = TwoModalityMLP()
train_supervised_nn(model, trainloader, valloader, task='classification', pos_weight=4.0, lr=1e-4, checkpoint_path='weights/mlp.pth', patience=2, device='cpu', epochs=10)

Epoch 1: 100%|██████████| 4634/4634 [00:08<00:00, 558.88it/s]


Epoch 1 | Val AUC: 0.8185
Saved new best model at epoch 1 (loss=0.8960)


Epoch 2: 100%|██████████| 4634/4634 [00:07<00:00, 590.86it/s]


Epoch 2 | Val AUC: 0.8219
Saved new best model at epoch 2 (loss=0.8553)


Epoch 3: 100%|██████████| 4634/4634 [00:07<00:00, 593.99it/s]


Epoch 3 | Val AUC: 0.8196


Epoch 4: 100%|██████████| 4634/4634 [00:08<00:00, 552.80it/s]


Epoch 4 | Val AUC: 0.8146
Early stopping triggered at epoch 4


In [17]:
from src import evaluate_nn, find_best_threshold

thresh = find_best_threshold(model, testloader, 'cpu')

evaluate_nn(model, testloader, thresh, 'cpu')

{'auc_score': 0.7263502511824911,
 'accuracy': 0.6590669014084507,
 'f1': 0.448526270824434,
 'precision': 0.3315091559671648,
 'recall': 0.6932218309859155}

# ChronoNet (EEG)

In [7]:
from src.models import ChronoNet

class ChronoNetConfig:
  fs = 256
  frame = 10
  CH = 2
  dropoutRate = 0.2
  l2 = 0.0001
  cnn_drop = 0.35
  batchnorm = True
  maxpool = False
  avgpool = False
  strided = False

cn = ChronoNet(ChronoNetConfig())

In [23]:
%load_ext autoreload
%autoreload 2

In [22]:
from src import train_supervised_nn

train_supervised_nn(cn, trainloader, valloader, task='classification', pos_weight=4.0, lr=1e-4, device='cpu', epochs=10)

Epoch 1:   0%|          | 14/4634 [00:13<1:15:11,  1.02it/s]


KeyboardInterrupt: 